# ADAUSDT intraminute segment features

???????? ?????? ????????? minute-level feature dataset ?????????????? ????????? ?? ?????? raw `aggTrades`, ?? ??????? raw-????.

Minute backbone ??????? ?? raw `klines`: ?????? ????? ? ???????? parquet ????? ????????? ??????? backbone ???????. ?????? ?????? ?????? ?????? ??????? ?? ???????? `[0, 20)`, `[20, 40)`, `[40, 60)`.

In [ ]:
import io
import os
import re

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv


BUCKET = "binance-data-downloader"
RAW_PREFIX = "raw"
FEATURES_PREFIX = "features"
FEATURE_DATASET_NAME = "intraminute_segments"
SYMBOL = "ADAUSDT"
INTERVAL = "1m"
SKIP_EXISTING = True

FEATURE_COLUMNS = [
    "open_time",
    "VWAP_minute",
    "VWAP_segment_1",
    "VWAP_segment_2",
    "VWAP_segment_3",
    "V_buy_segment_1",
    "V_sell_segment_1",
    "delta_segment_1",
    "pressure_segment_1",
    "V_buy_segment_2",
    "V_sell_segment_2",
    "delta_segment_2",
    "pressure_segment_2",
    "V_buy_segment_3",
    "V_sell_segment_3",
    "delta_segment_3",
    "pressure_segment_3",
]

FLOAT_FEATURE_COLUMNS = [column for column in FEATURE_COLUMNS if column != "open_time"]


def make_s3_client():
    load_dotenv()
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("YC_ENDPOINT"),
        region_name=os.getenv("YC_REGION"),
        aws_access_key_id=os.getenv("YC_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("YC_SECRET_ACCESS_KEY"),
    )

In [ ]:
def list_symbol_aggtrades_days(symbol: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, s3_client=None) -> list[str]:
    s3 = s3_client or make_s3_client()
    source_prefix = f"{raw_prefix.strip('/')}/aggTrades/symbol={symbol}/"
    pattern = re.compile(r"/date=(\d{4}-\d{2}-\d{2})/data\.parquet$")
    dates = set()
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=source_prefix):
        for obj in page.get("Contents", []):
            match = pattern.search(f"/{obj['Key']}")
            if match:
                dates.add(match.group(1))
    return sorted(dates)


def aggtrades_key(symbol: str, date: str, raw_prefix: str = RAW_PREFIX) -> str:
    return f"{raw_prefix.strip('/')}/aggTrades/symbol={symbol}/date={date}/data.parquet"


def klines_key(symbol: str, date: str, raw_prefix: str = RAW_PREFIX, interval: str = INTERVAL) -> str:
    return f"{raw_prefix.strip('/')}/klines/symbol={symbol}/interval={interval}/date={date}/data.parquet"


def feature_dataset_key(symbol: str, date: str, features_prefix: str = FEATURES_PREFIX, feature_dataset_name: str = FEATURE_DATASET_NAME, interval: str = INTERVAL) -> str:
    return f"{features_prefix.strip('/')}/{feature_dataset_name}/symbol={symbol}/interval={interval}/date={date}/data.parquet"


def s3_key_exists(s3_client, bucket: str, key: str) -> bool:
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except Exception as exc:
        error_code = getattr(exc, "response", {}).get("Error", {}).get("Code")
        if error_code in {"404", "NoSuchKey", "NotFound"}:
            return False
        raise

In [ ]:
def read_symbol_aggtrades_day(symbol: str, date: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, s3_client=None) -> pd.DataFrame:
    s3 = s3_client or make_s3_client()
    obj = s3.get_object(Bucket=bucket, Key=aggtrades_key(symbol, date, raw_prefix))
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    required_columns = ["transact_time", "price", "quantity", "is_buyer_maker"]
    missing_columns = sorted(set(required_columns) - set(df.columns))
    if missing_columns:
        raise ValueError(f"Missing required aggTrades columns: {missing_columns}")
    if df.empty:
        return pd.DataFrame(columns=required_columns)
    cleaned = df[required_columns].copy()
    cleaned["transact_time"] = pd.to_numeric(cleaned["transact_time"], errors="coerce").astype("Int64")
    cleaned["price"] = pd.to_numeric(cleaned["price"], errors="coerce")
    cleaned["quantity"] = pd.to_numeric(cleaned["quantity"], errors="coerce")
    cleaned["is_buyer_maker"] = cleaned["is_buyer_maker"].astype("boolean")
    return cleaned.dropna(subset=required_columns).reset_index(drop=True)


def read_minute_backbone_day(symbol: str, date: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, interval: str = INTERVAL, s3_client=None) -> pd.DataFrame:
    s3 = s3_client or make_s3_client()
    obj = s3.get_object(Bucket=bucket, Key=klines_key(symbol, date, raw_prefix, interval))
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=["open_time"])
    if df.empty:
        return pd.DataFrame(columns=["open_time"])
    backbone = df[["open_time"]].copy()
    backbone["open_time"] = pd.to_numeric(backbone["open_time"], errors="coerce").astype("Int64")
    backbone = backbone.dropna(subset=["open_time"]).drop_duplicates(subset=["open_time"]).sort_values("open_time").reset_index(drop=True)
    open_time_utc = pd.to_datetime(backbone["open_time"].astype("int64"), unit="ms", utc=True)
    if not open_time_utc.dt.second.eq(0).all() or not open_time_utc.dt.microsecond.eq(0).all():
        raise ValueError("Minute backbone open_time must be minute-aligned UTC")
    return backbone

In [ ]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    result = np.divide(
        numerator.astype("float64"),
        denominator.astype("float64"),
        out=np.zeros(len(numerator), dtype="float64"),
        where=denominator.astype("float64").to_numpy() != 0,
    )
    return pd.Series(result, index=numerator.index)


def prepare_trades_with_segments(trades: pd.DataFrame) -> pd.DataFrame:
    minute_bucket = (trades["transact_time"].astype("int64") // 60000) * 60000
    millisecond_offset = trades["transact_time"].astype("int64") % 60000
    segment = np.select(
        [millisecond_offset < 20000, millisecond_offset < 40000],
        [1, 2],
        default=3,
    ).astype("int8")
    sign = np.where(~trades["is_buyer_maker"], 1.0, -1.0)
    return trades.assign(
        open_time=minute_bucket.astype("int64"),
        millisecond_offset=millisecond_offset,
        segment=segment,
        quote_qty=trades["price"] * trades["quantity"],
        sign=sign,
        buy_qty=np.where(~trades["is_buyer_maker"], trades["quantity"], 0.0),
        sell_qty=np.where(trades["is_buyer_maker"], trades["quantity"], 0.0),
    )


def build_intraminute_segment_features_for_day(symbol: str, date: str, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, interval: str = INTERVAL, s3_client=None) -> pd.DataFrame:
    backbone = read_minute_backbone_day(symbol, date, bucket, raw_prefix, interval, s3_client)
    trades = read_symbol_aggtrades_day(symbol, date, bucket, raw_prefix, s3_client)
    if backbone.empty:
        return pd.DataFrame(columns=FEATURE_COLUMNS)
    if trades.empty:
        feature_df = backbone.copy()
        for column in FLOAT_FEATURE_COLUMNS:
            feature_df[column] = np.float32(0)
        return feature_df[FEATURE_COLUMNS]

    trades = prepare_trades_with_segments(trades)

    minute_agg = trades.groupby("open_time", as_index=False).agg(
        minute_quote_qty=("quote_qty", "sum"),
        minute_quantity=("quantity", "sum"),
    )
    minute_agg["VWAP_minute"] = safe_divide(minute_agg["minute_quote_qty"], minute_agg["minute_quantity"])
    trades = trades.merge(minute_agg[["open_time", "VWAP_minute"]], on="open_time", how="left")
    trades["pressure_component"] = trades["sign"] * trades["quantity"] * (trades["price"] - trades["VWAP_minute"])

    segment_agg = trades.groupby(["open_time", "segment"], as_index=False).agg(
        segment_quote_qty=("quote_qty", "sum"),
        segment_quantity=("quantity", "sum"),
        V_buy_segment=("buy_qty", "sum"),
        V_sell_segment=("sell_qty", "sum"),
        pressure_segment=("pressure_component", "sum"),
    )
    segment_agg["VWAP_segment"] = safe_divide(segment_agg["segment_quote_qty"], segment_agg["segment_quantity"])
    segment_agg["delta_segment"] = segment_agg["V_buy_segment"] - segment_agg["V_sell_segment"]

    wide = segment_agg.pivot(index="open_time", columns="segment", values=["VWAP_segment", "V_buy_segment", "V_sell_segment", "delta_segment", "pressure_segment"])
    wide.columns = [f"{name}_{segment}" for name, segment in wide.columns]
    wide = wide.reset_index()

    feature_df = backbone.merge(minute_agg[["open_time", "VWAP_minute"]], on="open_time", how="left")
    feature_df = feature_df.merge(wide, on="open_time", how="left")
    feature_df = feature_df.replace([np.inf, -np.inf], np.nan).fillna(0)

    for column in FLOAT_FEATURE_COLUMNS:
        if column not in feature_df.columns:
            feature_df[column] = 0
        feature_df[column] = feature_df[column].astype("float32")

    return feature_df[FEATURE_COLUMNS].sort_values("open_time").reset_index(drop=True)

In [ ]:
def write_intraminute_segments_for_symbol_to_s3(symbol: str = SYMBOL, bucket: str = BUCKET, raw_prefix: str = RAW_PREFIX, features_prefix: str = FEATURES_PREFIX, feature_dataset_name: str = FEATURE_DATASET_NAME, interval: str = INTERVAL, skip_existing: bool = SKIP_EXISTING, s3_client=None) -> pd.DataFrame:
    s3 = s3_client or make_s3_client()
    dates = list_symbol_aggtrades_days(symbol, bucket, raw_prefix, s3)
    if not dates:
        raise FileNotFoundError(f"No aggTrades days found for symbol={symbol}")
    rows=[]
    for date in dates:
        key = feature_dataset_key(symbol, date, features_prefix, feature_dataset_name, interval)
        if skip_existing and s3_key_exists(s3, bucket, key):
            print(f"Skip exists: s3://{bucket}/{key}")
            rows.append({"date": date, "rows": None, "key": key, "status": "skipped"})
            continue
        feature_df = build_intraminute_segment_features_for_day(symbol, date, bucket, raw_prefix, interval, s3)
        buffer = io.BytesIO()
        feature_df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
        s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
        print(f"Uploaded: s3://{bucket}/{key} rows={len(feature_df)}")
        rows.append({"date": date, "rows": len(feature_df), "key": key, "status": "uploaded"})
    return pd.DataFrame(rows)

In [ ]:
# Preflight: ???????????? ????????????? raw aggTrades ? ????????? config.yaml.
from config_loader import load_config
from aggtrades_backfill import backfill_missing_aggtrades_dates

config = load_config("config.yaml")
s3 = make_s3_client()
aggtrades_backfill_results = []

for symbol in config["symbols"]:
    aggtrades_backfill_results.append(
        backfill_missing_aggtrades_dates(
            symbol=symbol,
            start_date=config["date_range"]["start"],
            end_date=config["date_range"]["end"],
            bucket=config["storage"]["bucket"],
            raw_prefix=config["storage"]["prefix"],
            retries=config["download"]["retries"],
            timeout=tuple(config["download"]["timeout"]),
            s3_client=s3,
        )
    )

aggtrades_backfill_results = (
    pd.concat(aggtrades_backfill_results, ignore_index=True)
    if aggtrades_backfill_results
    else pd.DataFrame(columns=["symbol", "date", "rows", "key", "status"])
)

display(aggtrades_backfill_results)
if not aggtrades_backfill_results.empty:
    print(aggtrades_backfill_results["status"].value_counts(dropna=False))
else:
    print("No missing raw aggTrades days found.")

In [ ]:
# ??????? ???????? ?? ????? ??? ????? ???????? ???????.
s3 = make_s3_client()
dates = list_symbol_aggtrades_days(SYMBOL, s3_client=s3)
example_date = dates[1]
example_features = build_intraminute_segment_features_for_day(SYMBOL, example_date, s3_client=s3)

print(example_date, example_features.shape)
display(example_features.head())
display(example_features.tail())
display(example_features.dtypes)
print("zero-minute rows:", int((example_features[FLOAT_FEATURE_COLUMNS].sum(axis=1) == 0).sum()))

In [ ]:
# ???????? ?????? ???? ??????? parquet-?????? ????????? ? S3.
# ??????????????, ????? ?????? ????? ????????? ?????? ????????.
# write_results = write_intraminute_segments_for_symbol_to_s3(s3_client=s3)
# display(write_results)

In [ ]:
# ?????? ?? ???? ?????? ?? config.yaml.
from config_loader import load_config

config = load_config("config.yaml")
config_symbols = config["symbols"]
config_interval = config["interval"]
config_start_date = pd.Timestamp(config["date_range"]["start"]).date()
config_end_date = pd.Timestamp(config["date_range"]["end"]).date()
config_bucket = config["storage"]["bucket"]
config_raw_prefix = config["storage"]["prefix"]

s3 = make_s3_client()
all_write_results = []

for symbol in config_symbols:
    available_dates = list_symbol_aggtrades_days(
        symbol=symbol,
        bucket=config_bucket,
        raw_prefix=config_raw_prefix,
        s3_client=s3,
    )
    selected_dates = [
        date
        for date in available_dates
        if config_start_date <= pd.Timestamp(date).date() <= config_end_date
    ]

    if not selected_dates:
        print(f"No aggTrades days found in config range for symbol={symbol}")
        continue

    rows = []
    for date in selected_dates:
        key = feature_dataset_key(
            symbol=symbol,
            date=date,
            interval=config_interval,
        )

        if SKIP_EXISTING and s3_key_exists(s3, config_bucket, key):
            print(f"Skip exists: s3://{config_bucket}/{key}")
            rows.append({"symbol": symbol, "date": date, "rows": None, "key": key, "status": "skipped"})
            continue

        feature_df = build_intraminute_segment_features_for_day(
            symbol=symbol,
            date=date,
            bucket=config_bucket,
            raw_prefix=config_raw_prefix,
            interval=config_interval,
            s3_client=s3,
        )

        buffer = io.BytesIO()
        feature_df.to_parquet(buffer, index=False, engine="pyarrow", compression="zstd")
        s3.put_object(Bucket=config_bucket, Key=key, Body=buffer.getvalue())

        print(f"Uploaded: s3://{config_bucket}/{key} rows={len(feature_df)}")
        rows.append({"symbol": symbol, "date": date, "rows": len(feature_df), "key": key, "status": "uploaded"})

    all_write_results.append(pd.DataFrame(rows))

write_results_config_period = (
    pd.concat(all_write_results, ignore_index=True)
    if all_write_results
    else pd.DataFrame(columns=["symbol", "date", "rows", "key", "status"])
)

display(write_results_config_period)
print(write_results_config_period["status"].value_counts(dropna=False))